# ChromaDB Inspector

Query and debug ChromaDB collections for each chunking strategy.

**Collections:**
- `squad_baseline_258_tok` → `./squad_chroma_db_baseline_258_tok`
- `squad_most_recent_low_acc_258t_w128_wtd` → `./squad_chroma_db_most_recent_low_acc_258t_w128_wtd`
- `squad_most_recent_258t_w254` → `./squad_chroma_db_most_recent_258t_w254`
- `squad_nonlinear_258t_w254` → `./squad_chroma_db_nonlinear_258t_w254`

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve().parents[0]
sys.path.append(str(REPO_ROOT))

import chromadb
from sentence_transformers import SentenceTransformer
from index_documents import LocalEmbeddingFunction
from config import EMBEDDING_MODEL_NAME

STRATEGIES = [
    'baseline_258_tok',
    'most_recent_low_acc_258t_w128_wtd',
    'most_recent_258t_w254',
    'nonlinear_258t_w254',
]

print(f'Embedding model: {EMBEDDING_MODEL_NAME}')
model = SentenceTransformer(EMBEDDING_MODEL_NAME)
embed_fn = LocalEmbeddingFunction(model)
print('Model loaded.')

/projects/tejo9855/software/anaconda/envs/teagan-conda-env-curc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/projects/tejo9855/software/anaconda/envs/teagan-conda-env-curc/lib/python3.11/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:06<00:00, 16.91it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


In [16]:
def get_collection(strategy: str) -> chromadb.Collection:
    client = chromadb.PersistentClient(
        path=str(REPO_ROOT / f'squad_chroma_db_{strategy}')
    )
    return client.get_collection(
        name=f'squad_{strategy}',
        embedding_function=embed_fn,
    )

## Collection Sizes

In [19]:
for strategy in STRATEGIES:
    try:
        col = get_collection(strategy)
        print(f'{strategy:45s} {col.count():>6} chunks')
    except Exception as e:
        print(f'{strategy:45s} ERROR: {e}')

baseline_258_tok                               18588 chunks
most_recent_low_acc_258t_w128_wtd              18533 chunks
most_recent_258t_w254                         ERROR: Database error: error returned from database: (code: 3850) disk I/O error
nonlinear_258t_w254                           ERROR: Database error: error returned from database: (code: 3850) disk I/O error


In [26]:
def get_all_sources(col, batch_size=5000):
    sources = set()
    offset = 0
    while True:
        batch = col.get(limit=batch_size, offset=offset, include=['metadatas'])
        if not batch['metadatas']:
            break
        sources.update(m['source'] for m in batch['metadatas'])
        offset += batch_size
    return sources

for strategy in STRATEGIES:
    try:
        col = get_collection(strategy)
        sources = get_all_sources(col)
        print(f'{strategy}: {len(sources)} articles, {col.count()} chunks')
    except Exception as e:
        print(f'{strategy}: ERROR {e}')

baseline_258_tok: 412 articles, 18588 chunks
most_recent_low_acc_258t_w128_wtd: 412 articles, 18533 chunks
most_recent_258t_w254: 412 articles, 20838 chunks
nonlinear_258t_w254: 412 articles, 23432 chunks


In [9]:
import json                                                                                                                                                       
from pathlib import Path                                                                                                                                          

for path in sorted(Path('squad_testing/output').glob('evaluated_squad_*.jsonl')):                                                                                 
    total = 0                                             
    not_found = 0                                                                                                                                                 
    with open(path) as f:                                 
        for line in f:
            if line.strip():
                obj = json.loads(line)
                total += 1
                print(obj.get('note'))
                if obj.get('note') == 'article not found in corpus':
                    not_found += 1                                                                                                                                
    print(f'{path.name}: {not_found}/{total} not found')  

In [24]:
col = get_collection('most_recent_258t_w254')                                                                                                                     
all_meta = col.get(include=['metadatas'])                                                                                                                         
sources = set(m['source'] for m in all_meta['metadatas'])                                                                                                         
# Find anything with 'high' or 'television' in the name                                                                                                           
matches = [s for s in sources if 'igh' in s.lower() or 'television' in s.lower()]                                                                                 
print(matches) 

['Daylight saving time.txt', 'Lighting.txt', 'BBC Television.txt', 'The Legend of Zelda  Twilight Princess.txt', 'Brigham Young University.txt', 'Age of Enlightenment.txt', 'High definition television.txt', 'Light emitting diode.txt', 'Incandescent light bulb.txt', 'Dwight D  Eisenhower.txt', 'Copyright infringement.txt', 'CBC Television.txt', 'Raleigh  North Carolina.txt']


## Peek at Records

Inspect the first N records in a collection — useful for verifying source name format.

In [ ]:
STRATEGY = 'baseline_258_tok'  # change to inspect a different strategy
N        = 10

col = get_collection(STRATEGY)
results = col.get(limit=N, include=['metadatas', 'documents'])

for meta, doc in zip(results['metadatas'], results['documents']):
    print(f"{meta}")
    print(f"  {repr(doc[:80])}")
    print()

## All Unique Source Names

Lists every distinct `source` value stored in a collection.
Compare against what `evaluate_squad.py` constructs to diagnose "article not found" mismatches.

In [10]:
STRATEGY = 'baseline_258_tok'  # change to inspect a different strategy

col = get_collection(STRATEGY)
all_meta = col.get(include=['metadatas'])
sources = sorted(set(m['source'] for m in all_meta['metadatas']))

print(f'{len(sources)} unique sources in {STRATEGY}:')
for s in sources[:30]:
    print(f'  {s}')
if len(sources) > 30:
    print(f'  ... ({len(sources) - 30} more)')

412 unique sources in baseline_258_tok:
  2008 Sichuan earthquake.txt
  2008 Summer Olympics torch relay.txt
  51st state.txt
  A cappella.txt
  ASCII.txt
  Adolescence.txt
  Adult contemporary music.txt
  Affirmative action in the United States.txt
  Age of Enlightenment.txt
  Aircraft carrier.txt
  Airport.txt
  Alaska.txt
  Alexander Graham Bell.txt
  Alfred North Whitehead.txt
  Alloy.txt
  Alps.txt
  Alsace.txt
  American Idol.txt
  Animal.txt
  Ann Arbor, Michigan.txt
  Annelid.txt
  Antarctica.txt
  Antenna (radio).txt
  Anthropology.txt
  Anti-aircraft warfare.txt
  Apollo.txt
  Appalachian Mountains.txt
  Architecture.txt
  Arena Football League.txt
  Armenia.txt
  ... (382 more)


## Look Up a Specific Article

Check whether a particular article exists in a collection and inspect its chunks.

In [ ]:
STRATEGY     = 'baseline_258_tok'
ARTICLE_NAME = '2008 Sichuan earthquake.txt'  # must match source format exactly

col = get_collection(STRATEGY)
results = col.get(
    where={'source': {'$eq': ARTICLE_NAME}},
    include=['metadatas', 'documents'],
)

print(f'Found {len(results["ids"])} chunks for "{ARTICLE_NAME}" in {STRATEGY}')
for meta, doc in zip(results['metadatas'], results['documents']):
    print(f"  chunk {meta['chunk_index']:3d}: {repr(doc[:80])}")

## Find Missing Articles Across Strategies

Compares source names across all four collections to find articles present in some but not others.

In [ ]:
import pandas as pd

strategy_sources = {}
for strategy in STRATEGIES:
    try:
        col = get_collection(strategy)
        all_meta = col.get(include=['metadatas'])
        strategy_sources[strategy] = set(m['source'] for m in all_meta['metadatas'])
    except Exception as e:
        print(f'Could not load {strategy}: {e}')
        strategy_sources[strategy] = set()

all_sources = set.union(*strategy_sources.values())
rows = []
for source in sorted(all_sources):
    row = {'source': source}
    for strategy in STRATEGIES:
        row[strategy[:12]] = '✓' if source in strategy_sources[strategy] else '✗'
    rows.append(row)

df = pd.DataFrame(rows)
missing = df[(df.iloc[:, 1:] == '✗').any(axis=1)]
print(f'{len(missing)} articles missing from at least one strategy:')
display(missing.reset_index(drop=True))

## Test Retrieval

Run a retrieval query against a collection to see what chunks come back — the same call `evaluate_squad.py` makes.

In [ ]:
STRATEGY     = 'baseline_258_tok'
QUESTION     = 'What fault did the 2008 Sichuan earthquake occur on?'
ARTICLE_NAME = '2008 Sichuan earthquake.txt'  # set to None for global retrieval
N_RESULTS    = 10

col = get_collection(STRATEGY)
kwargs = dict(
    query_texts=[QUESTION],
    n_results=N_RESULTS,
    include=['documents', 'metadatas', 'distances'],
)
if ARTICLE_NAME:
    kwargs['where'] = {'source': {'$in': [ARTICLE_NAME]}}

results = col.query(**kwargs)

print(f'Query: {QUESTION!r}')
print(f'Filter: {ARTICLE_NAME or "none (global)"}')
print()
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0],
)):
    print(f'[{i+1}] chunk={meta["chunk_index"]}  distance={dist:.4f}')
    print(f'     {repr(doc[:120])}')
    print()